In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objs as go
from plotly.subplots import make_subplots
from datetime import timedelta

import requests
from bs4 import BeautifulSoup

from scripts.preparation import download_data, add_macd, add_moving_average, add_psar

In [8]:
def plot_signal(stock_data, centroid_diff, title):
    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.02,
        row_width=[0.4, 0.6],
    )

    candlestick = go.Candlestick(
        x=stock_data.index,
        open=stock_data["Open"],
        high=stock_data["High"],
        low=stock_data["Low"],
        close=stock_data["Close"],
        name="Candles",
    )

    fig.add_trace(
        candlestick,
        row=1,
        col=1,
    )

    for ema in ["ema5", "ema20", "ema60"]:
        fig.add_trace(
            go.Scatter(
                x=stock_data.index,
                y=stock_data[ema],
                mode="lines",
                name=ema,
            ),
            row=1,
            col=1,
        )

    fig.add_trace(
        go.Bar(
            x=stock_data.index,
            y=stock_data["centroid_diff"],
            name="centroid_diff",
        ),
        row=2,
        col=1,
    )

    for line_plot in ["macd_ma3", "macd_ma5"]:
        fig.add_trace(
            go.Scatter(
                x=stock_data.index,
                y=stock_data[line_plot],
                mode="lines",
                name=line_plot,
            ),
            row=2,
            col=1,
        )
    
    days = 40

    args = {
            "xaxis.range": [
                stock_data.index[-days],
                stock_data.index[-1] + timedelta(5),
            ],
            "yaxis.range": [
                (0.9 * min(stock_data.iloc[-days:]["Low"])),
                (1.1 * max(stock_data.iloc[-days:]["High"])),
            ]
        }
    
    fig.update_layout(
        autosize=True,
        xaxis_rangeslider_visible=False,
        height=800,
        title={"text": title},
        **args
    )

    fig.show()


# screener testing

In [2]:
def extract_ticker(df_row, russell_list):
    capital = 0
    for letter in df_row:
        if letter.isupper():
            capital += 1
        else:
            break

    
    answer = df_row[:capital-1]
    debug = answer
    white_list = ["SMCI", "MSTR"]

    while len(answer) > 0:
        if answer in (russell_list + white_list):
            return answer
        else:
            answer = answer[:len(answer)-1]
    
    print("Can't find: ", debug)
    return None



# Ticker choice

In [3]:
sp500_table = pd.read_html("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies")
sp500_df = sp500_table[0][["Symbol", "GICS Sector"]]
sp500_df["GICS Sector"].unique()

array(['Industrials', 'Health Care', 'Information Technology',
       'Utilities', 'Financials', 'Materials', 'Consumer Discretionary',
       'Real Estate', 'Communication Services', 'Consumer Staples',
       'Energy'], dtype=object)

In [4]:
# ticker_list = list(sp500_df[sp500_df["GICS Sector"] == "Utilities"]["Symbol"])
ticker_list = list(i for i in sp500_df["Symbol"] if "." not in i)

In [5]:
russell_table = pd.read_html("https://en.wikipedia.org/wiki/Russell_1000_Index")
russell_list = list(russell_table[2]["Ticker"])


top100 = pd.read_html("https://www.tradingview.com/markets/stocks-usa/market-movers-active/")
ticker_list = list(top100[0]["Symbol"].apply(extract_ticker, russell_list=russell_list))
ticker_list = [i for i in ticker_list if i is not None]


In [6]:
ticker_object = download_data(ticker_list)
print_all = False

In [9]:
holding = "SHOP PINS ABNB"
holding_list = holding.split()

ticker_list = list(set(holding_list))

ticker_object = download_data(ticker_list)
print_all = True

In [10]:
count = 0
for ticker in ticker_list:
    stock_data = ticker_object.tickers[ticker].history(period="200d")
    stock_data = add_macd(stock_data)
    stock_data = add_moving_average(stock_data)
    stock_data = add_psar(stock_data)
    stock_data["ema_bull"] = True
    for i in ["ema5", "ema20", "ema60"]:
        stock_data[f"{i}_bull"] = stock_data[i] > stock_data[i].shift(1)
        stock_data["ema_bull"] *= stock_data[f"{i}_bull"]

    stock_data["centroid"] = (stock_data["Open"] + stock_data["Close"]) / 2
    stock_data["centroid_diff"] = stock_data["centroid"].pct_change()



    if print_all:
        print(ticker, count)
        plot_signal(stock_data, stock_data["centroid_diff"], ticker)

    # # if (sum(macd_up_idx.tail(3)) > 0 and sum(macd_down_idx.tail(3)) == 0 and sum(turn_idx.tail(5)) < 4):
    # else:
    #     if ((sum(macd_up_idx.tail(2)) > 0 
    #         or sum(turn_idx.tail(2)) > 0)
    #         and sum(macd_down_idx.tail(2)) == 0):
    #         print(ticker, count)
    #         plot_signal(stock_data, turn_idx, macd_up_idx, macd_down_idx, ticker)
    #         count += 1

    # if count > 10:
    #     break
# print(count)


PINS 0


SHOP 0


ABNB 0


# End